# Evaluation of Fine-Tuned Model (Full):

This Fine-Tuned model was taken from Course Instructor's HF Repo, Trained by him.

## Imports and Constants:

In [1]:
# Imports:
import os
from dotenv import load_dotenv
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from datetime import datetime
from peft import PeftModel
from pricer.util import evaluate

In [2]:
# Constants:
BASE_MODEL = 'meta-llama/Llama-3.2-3B'
PROJECT_NAME = 'price'
HF_USER = 'ed-donner'
DATASET_NAME = f'{HF_USER}/items_prompts_full'


# Using Adapters from RUN-2, at Training Step 500, just before overfitting started:
RUN_NAME ='2025-11-28_18.47.07'
REVISION = 'b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd'

PROJECT_RUN_NAME = f'{PROJECT_NAME}-{RUN_NAME}'
HUB_MODEL_NAME = f'{HF_USER}/{PROJECT_RUN_NAME}'

In [3]:
# Hyper-Parameters for QLORA:
QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8
print(use_bf16)

True


## Login to Hugging Face:

In [4]:
load_dotenv(override= True)
hf_token = os.getenv('HF_TOKEN')
login(token= hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Loading Dataset:

In [6]:
dataset = load_dataset(path= DATASET_NAME, split= 'test')

In [7]:
print(f'Loaded {len(dataset)} Test Items')

Loaded 10000 Test Items


## Loading Tokenizer and Model:

In [8]:
# Quantization Configuration:
if QUANT_4_BIT:
    quant_config= BitsAndBytesConfig(
        load_in_4bit= True,
        bnb_4bit_quant_type= 'nf4',
        bnb_4bit_use_double_quant= True,
        bnb_4bit_compute_dtype= torch.bfloat16 if use_bf16 else torch.float16
    )
else:
    quant_config= BitsAndBytesConfig(
        load_in_8bit= True,
        bnb_4bit_compute_dtype= torch.bfloat16 if use_bf16 else torch.float16
    )

In [9]:
# Loading Tokenizer:
tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path= BASE_MODEL,
    trust_remote_code= True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side= 'right'

# Model:
base_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path= BASE_MODEL,
    quantization_config= quant_config,
    device_map= 'auto'
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

## Loading Fine-Tuned Model with Peft:

In [10]:
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    HUB_MODEL_NAME,
    revision= REVISION
)

print(f'Memory Foot Print of Base Model + Adapters: {fine_tuned_model.get_memory_footprint() / 1e9:.2f}GB')

adapter_model.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

C:\Users\shail\anaconda3\envs\applied_llm_engineering\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shail\.cache\huggingface\hub\models--ed-donner--price-2025-11-28_18.47.07. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Memory Foot Print of Base Model + Adapters: 3.75GB


## Architecture of Fine-Tuned Model:

In [11]:
fine_tuned_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=256, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=256, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lo

## Function to Predict Price of Test Item:

In [12]:
def model_predict(item):
    inputs = tokenizer(item['prompt'], return_tensors='pt').to('cuda')

    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens= 8)

    prompt_len = inputs['input_ids'].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [14]:
model_predict(dataset[0])

'209.00'

## Fine-Tuned Model Evaluation:

In [15]:
evaluate(function= model_predict,
         data= dataset,
         size= 200)

![Evaluation Chart](../assets/fine-tuned(full).jpg)